# ⚙️ SÍNTESE 2 — FASE DE OTIMIZAÇÃO
## Notebooks 11–15 | Frentes 1–5 | Evidências Reais

---
Frentes de melhoria aplicadas ao pipeline base para aumentar pAUC@0.1
em cenários de domain shift e generalização cross-domain.


In [ ]:
import json, sys
sys.path.insert(0, '..')
from pathlib import Path
METRICS_DIR = Path('experiments_results/metrics')
def load(nb_id): p = METRICS_DIR/f'{nb_id}_results.json'; return json.load(open(p)) if p.exists() else {}
nb11=load('nb11'); nb12=load('nb12'); nb13=load('nb13'); nb14=load('nb14'); nb15=load('nb15')
print("✅ Resultados NB11-15 carregados")


## Frente 1 — NB11: Regularização L2 + Limiar Gamma

In [ ]:
print("NB11 — Regularização L2")
for k, v in nb11.get('models',{}).items():
    if isinstance(v, dict):
        print(f"  {v.get('label','?'):15s} pAUC={v.get('pauc01',0):.4f}")
print(f"\nLimiar Gamma calibrado: {nb11.get('gamma_threshold', '?'):.4f}")


## Frente 2 — NB12: GMM Campeão Não Supervisionado

In [ ]:
print("NB12 — GMM vs Mahalanobis vs CNN")
for k, v in nb12.get('models',{}).items():
    if isinstance(v, dict):
        lat = v.get('latency_ms', '-')
        print(f"  {v.get('label','?'):25s} pAUC={v.get('pauc01',0):.4f} | "
              f"F1={v.get('f1',0):.4f} | Lat={lat:.2f}ms" if isinstance(lat,float)
              else f"  {v.get('label','?'):25s} pAUC={v.get('pauc01',0):.4f}")


## Frente 3 — NB13: HHT+UKF ← Componente Mais Crítico

In [ ]:
print("NB13 — Impacto do HHT+UKF")
raw = nb13.get('models',{}).get('raw',{})
hht = nb13.get('models',{}).get('hht',{})
gain = nb13.get('pauc_gain', 0)
snr  = nb13.get('snr_gain_db', 0)
print(f"  Sem HHT+UKF: pAUC={raw.get('pauc01',0):.4f}")
print(f"  Com HHT+UKF: pAUC={hht.get('pauc01',0):.4f}")
print(f"  Ganho pAUC: +{gain:.4f} pp")
print(f"  Ganho SNR:  +{snr:.1f} dB")


## Frente 4 — NB14: Mixup para Domain Shift

In [ ]:
print("NB14 — Mixup (α=0.4)")
no_mix = nb14.get('models',{}).get('no_mixup',{})
mix    = nb14.get('models',{}).get('mixup',{})
gain   = nb14.get('pauc_gain', 0)
print(f"  Sem Mixup: pAUC={no_mix.get('pauc01',0):.4f}")
print(f"  Com Mixup: pAUC={mix.get('pauc01',0):.4f}")
pct = (gain / (no_mix.get('pauc01',1)+1e-10)) * 100
print(f"  Melhoria:  +{gain:.4f} pp ({pct:.1f}%)")


## Frente 5 — NB15: pAUC@0.1 como Métrica Principal

In [ ]:
print("NB15 — Ranking por pAUC@0.1 vs F1 global")
ranking = nb15.get('ranking', {})
print(f"{'Modelo':15s} | {'pAUC@0.1':10s} | {'F1':8s} | {'AUC':8s}")
print("-" * 50)
for name, met in sorted(ranking.items(), key=lambda x: -x[1].get('pauc01',0)):
    print(f"  {name:15s} | {met.get('pauc01',0):.4f}     | {met.get('f1',0):.4f}   | {met.get('auc',0):.4f}")
print("\n⚠️  Tiny-AST lidera em F1/AUC mas fica atrás em pAUC@0.1")
